In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
from scipy import stats
from tqdm import tqdm
from MDAnalysis import Universe

/home/bala_nsm/anaconda3/envs/mdan/lib/python3.11/site-packages/MDAnalysis/topology/TPRParser.py:161: DeprecationWarning: 'xdrlib' is deprecated and slated for removal in Python 3.13
  import xdrlib


In [2]:
TOPOLOGY = "../../3-nvt/nvt.tpr"
TRAJECTORY = "../../3-nvt/nvt_nojump.xtc"
CAT_SEL = "resname LIP"
AN_SEL = "resname _PF"
Q_CAT, Q_AN = +0.8, -0.8
START, END = 40000, 80000
DT_PS = 1.0
TEMP = 298.0

In [3]:
def autocorr_fft(x):
    N = len(x)
    F = np.fft.fft(x, n=2 * N)
    res = np.fft.ifft(F * F.conjugate())[:N].real
    return res / (N - np.arange(N))

def msd_fft(r):
    N = len(r)
    D = np.append(np.square(r).sum(axis=1), 0.0)
    S2 = sum(autocorr_fft(r[:, i]) for i in range(r.shape[1]))
    k = np.arange(N)
    Q = 2 * D.sum() - np.cumsum(D[k - 1] + D[N - k])
    S1 = Q / (N - k)
    return S1 - 2 * S2

def get_beta(msd, t, start, end):
    s = np.gradient(np.log(msd[start:end]), np.log(t[start:end]))
    return np.mean(s), np.max(s) - np.min(s)

def choose_msd_fitting_region(msd, t):
    beta_best = 0
    s_f, e_f = 2, len(t) // 10

    for i in np.logspace(np.log10(2), np.log10(len(t) / 10), 10):
        start, end = int(i), int(i * 10)

        if end >= len(t):
            continue

        if np.all(msd[start:end] > 0):
            beta, beta_range = get_beta(msd, t, start, end)

            if (abs(beta - 1) < abs(beta_best - 1) and beta_range < 2) or beta_best == 0:
                beta_best, s_f, e_f = beta, start, end

    return s_f, e_f, beta_best


def conductivity(msd, sl):
    fit = stats.linregress(time_ps[sl], msd[sl])
    return fit.slope * to_S_per_m, fit.stderr * to_S_per_m

In [4]:
u = Universe(TOPOLOGY, TRAJECTORY)
cat = u.select_atoms(CAT_SEL)
an = u.select_atoms(AN_SEL)

n_cat, n_an = len(cat.residues), len(an.residues)
atoms_cat = len(cat) // n_cat
atoms_an = len(an) // n_an

w_cat = cat.masses.reshape(n_cat, atoms_cat, 1)
w_an = an.masses.reshape(n_an, atoms_an, 1)

n_frames = END - START
R_cat = np.empty((n_frames, n_cat, 3))
R_an = np.empty((n_frames, n_an, 3))

for k, ts in enumerate(tqdm(u.trajectory[START:END], total=n_frames)):
    pos_c = cat.positions.astype(np.float64).reshape(n_cat, atoms_cat, 3)
    pos_a = an.positions.astype(np.float64).reshape(n_an, atoms_an, 3)

    R_cat[k] = (pos_c * w_cat).sum(1) / w_cat.sum(1)
    R_an[k] = (pos_a * w_an).sum(1) / w_an.sum(1)


Z_cat = Q_CAT * R_cat.sum(axis=1)
Z_an = Q_AN * R_an.sum(axis=1)

msd_total = msd_fft(Z_cat + Z_an)
msd_cc = msd_fft(Z_cat)
msd_aa = msd_fft(Z_an)

terms = {
    "Total": msd_total,
    "Li-Li collective": msd_cc,
    "PF6-PF6 collective": msd_aa, 
}

time_ps = np.arange(n_frames) * DT_PS

Lx, Ly, Lz = u.trajectory[0].dimensions[:3]
V = Lx * Ly * Lz

e = 1.60217662e-19
kB = 1.38064852e-23

to_S_per_m = (
    e * e * 1e-20 / 1e-12 /
    (kB * TEMP * V * 1e-30) / 6.0
)

results = {}

for name, msd in terms.items():
    start, end, beta = choose_msd_fitting_region(msd, time_ps)
    sl = slice(start, end)

    sigma, error = conductivity(msd, sl)
    results[name] = (sigma, error)

    print(f"\n{name}")
    print(f"Fit interval : {time_ps[start]:.0f} - {time_ps[end-1]:.0f} ps")
    print(f"Beta         : {beta:.3f}")
    print(f"Conductivity : {sigma*10:.4f} +/- {error*10:.4f} mS/cm")

sigma_tot, err_tot = results["Total"]
sigma_cc, err_cc = results["Li-Li collective"]
sigma_aa, err_aa = results["PF6-PF6 collective"]
sigma_cross = sigma_tot - sigma_cc - sigma_aa

print("\nLi-PF6 cross")
print(f"Conductivity : {sigma_cross*10:.4f}")

100%|████████████████████████████████| 40000/40000 [01:18<00:00, 509.50it/s]



Total
Fit interval : 136 - 1363 ps
Beta         : 0.951
Conductivity : 9.6613 +/- 0.0047 mS/cm

Li-Li collective
Fit interval : 136 - 1363 ps
Beta         : 0.989
Conductivity : 5.3037 +/- 0.0057 mS/cm

PF6-PF6 collective
Fit interval : 58 - 585 ps
Beta         : 0.997
Conductivity : 9.4480 +/- 0.0089 mS/cm

Li-PF6 cross
Conductivity : -5.0905
